In [ ]:
# Jakob Eisenhauer


import numpy as np
import pandas as pd

from scipy.special import gamma
from scipy.optimize import minimize

# Load data
url = 'https://raw.githubusercontent.com/luchem/bern02/main/Labs/bird_count.csv'
data = pd.read_csv(url)

counts = np.array(data['count'])
years = np.array(data['yr']) - 1999


# Poisson regression model
def model(beta, x):

    beta0, beta1 = beta

    lam = np.exp(beta0 + beta1 * x)

    return lam


# Negative log-likelihood
def minus_log_likelihood_poisson(beta, years, counts):
    '''
    Log-likelihood for a Poisson distribution:

    The likelihood is the product of the individual probabilities.
    The log-likelihood is therefore the sum of the log-probabilities.

    To maximize the likelihood, we minimize the negative log-likelihood.
    '''

    lam = model(beta, years)

    return -np.sum(
        -lam
        + counts * np.log(lam)
        - np.log(gamma(counts + 1))
    )


# Fit model
result = minimize(
    minus_log_likelihood_poisson,
    x0=[0, 0],
    args=(years, counts)
)

beta = result.x
print("Estimated beta:", beta)


# Expected count for each year
lambdas = model(beta, years)


# Simulate three hypothetical datasets
sim_results = []

for i in range(1, 4):

    sim_result = np.random.poisson(lambdas)

    result = pd.DataFrame({
        'yr': years + 1999,
        'count': sim_result,
        'sample': i})

    sim_results.append(result)

sim_results = pd.concat(sim_results)

sim_results = sim_results.sort_values(by=['sample','yr']).reset_index(drop=True)

sim_results.to_csv('simulated_bird_counts.csv', index=False)



#print(sim_results)

Estimated beta: [ 2.32533196 -0.03244318]
